# Bahdanau ???
:label:`sec_seq2seq_attention`

 :numref:`sec_seq2seq`??????????:
????????????????????-?????,
??????????
????,??????????????????????????????,
????????????????????????
???????(??)?????
??,????????(?)?????????????,
?????????????*??*???????
???????????????

????? :cite:`Graves.2013`?????:
????????????????,
Graves????????????,
?????????????,
???????????????
??????????,Bahdanau??????????????????
??????? :cite:`Bahdanau.Cho.Bengio.2014`?
??????,?????????????,??????(???)?????????????????????????????????????????

## ??

?????Bahdanau?????
??? :numref:`sec_seq2seq`?????????
????????????? :numref:`sec_seq2seq`??????,
??? :eqref:`eq_seq2seq_s_t`???????$\mathbf{c}$
????????$t'$???$\mathbf{c}_{t'}$???
????????$T$???,
?????$t'$???????????????:

$$\mathbf{c}_{t'} = \sum_{t=1}^T \alpha(\mathbf{s}_{t' - 1}, \mathbf{h}_t) \mathbf{h}_t,$$

??,???$t' - 1$????????$\mathbf{s}_{t' - 1}$???,
??????$\mathbf{h}_t$???,???,
?????$\alpha$??? :eqref:`eq_attn-scoring-alpha`
?????????????????

? :numref:`fig_seq2seq_details`???????????-?????????,
 :numref:`fig_s2s_attention_details`???Bahdanau???????

![????Bahdanau?????????????-?????](../img/seq2seq-attention-details.svg)
:label:`fig_s2s_attention_details`


In [ ]:
from mxnet import np, npx
from mxnet.gluon import nn, rnn
from d2l import mxnet as d2l

npx.set_np()

## ????????

????????Bahdanau???,???????????-????
??,??????????????
????????????????,
??`AttentionDecoder`????[**???????????????**]?


In [ ]:
#@save
class AttentionDecoder(d2l.Decoder):
    """???????????????"""
    def __init__(self, **kwargs):
        super(AttentionDecoder, self).__init__(**kwargs)

    @property
    def attention_weights(self):
        raise NotImplementedError

???,????????`Seq2SeqAttentionDecoder`??
[**????Bahdanau?????????????**]?
??,?????????,???????:

1. ????????????????,??????????;
1. ??????????????,?????????????;
1. ???????(????????????)?

??????????,??????????????????????
??,???????????????????????????


In [ ]:
class Seq2SeqAttentionDecoder(AttentionDecoder):
    def __init__(self, vocab_size, embed_size, num_hiddens, num_layers,
                 dropout=0, **kwargs):
        super(Seq2SeqAttentionDecoder, self).__init__(**kwargs)
        self.attention = d2l.AdditiveAttention(num_hiddens, dropout)
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.rnn = rnn.GRU(num_hiddens, num_layers, dropout=dropout)
        self.dense = nn.Dense(vocab_size, flatten=False)

    def init_state(self, enc_outputs, enc_valid_lens, *args):
        # outputs????(num_steps,batch_size,num_hiddens)
        # hidden_state[0]????(num_layers,batch_size,num_hiddens)
        outputs, hidden_state = enc_outputs
        return (outputs.swapaxes(0, 1), hidden_state, enc_valid_lens)

    def forward(self, X, state):
        # enc_outputs????(batch_size,num_steps,num_hiddens).
        # hidden_state[0]????(num_layers,batch_size,
        # num_hiddens)
        enc_outputs, hidden_state, enc_valid_lens = state
        # ??X????(num_steps,batch_size,embed_size)
        X = self.embedding(X).swapaxes(0, 1)
        outputs, self._attention_weights = [], []
        for x in X:
            # query????(batch_size,1,num_hiddens)
            query = np.expand_dims(hidden_state[0][-1], axis=1)
            # context????(batch_size,1,num_hiddens)
            context = self.attention(
                query, enc_outputs, enc_outputs, enc_valid_lens)
            # ????????
            x = np.concatenate((context, np.expand_dims(x, axis=1)), axis=-1)
            # ?x???(1,batch_size,embed_size+num_hiddens)
            out, hidden_state = self.rnn(x.swapaxes(0, 1), hidden_state)
            outputs.append(out)
            self._attention_weights.append(self.attention.attention_weights)
        # ???????,outputs????
        # (num_steps,batch_size,vocab_size)
        outputs = self.dense(np.concatenate(outputs, axis=0))
        return outputs.swapaxes(0, 1), [enc_outputs, hidden_state,
                                        enc_valid_lens]

    @property
    def attention_weights(self):
        return self._attention_weights

???,????7?????4?????????[**??Bahdanau??????**]?


In [ ]:
encoder = d2l.Seq2SeqEncoder(vocab_size=10, embed_size=8, num_hiddens=16,
                             num_layers=2)
encoder.initialize()
decoder = Seq2SeqAttentionDecoder(vocab_size=10, embed_size=8, num_hiddens=16,
                                  num_layers=2)
decoder.initialize()
X = np.zeros((4, 7))  # (batch_size,num_steps)
state = decoder.init_state(encoder(X), None)
output, state = decoder(X, state)
output.shape, len(state), state[0].shape, len(state[1]), state[1][0].shape

## [**??**]

? :numref:`sec_seq2seq_training`??,
??????????,???????Bahdanau???????????,
???????????????
??????????,????????????
 :numref:`sec_seq2seq_training`????


In [ ]:
embed_size, num_hiddens, num_layers, dropout = 32, 32, 2, 0.1
batch_size, num_steps = 64, 10
lr, num_epochs, device = 0.005, 250, d2l.try_gpu()

train_iter, src_vocab, tgt_vocab = d2l.load_data_nmt(batch_size, num_steps)
encoder = d2l.Seq2SeqEncoder(
    len(src_vocab), embed_size, num_hiddens, num_layers, dropout)
decoder = Seq2SeqAttentionDecoder(
    len(tgt_vocab), embed_size, num_hiddens, num_layers, dropout)
net = d2l.EncoderDecoder(encoder, decoder)
d2l.train_seq2seq(net, train_iter, lr, num_epochs, tgt_vocab, device)

?????,????[**????????????**]??????BLEU???


In [ ]:
engs = ['go .', "i lost .", 'he\'s calm .', 'i\'m home .']
fras = ['va !', 'j\'ai perdu .', 'il est calme .', 'je suis chez moi .']
for eng, fra in zip(engs, fras):
    translation, dec_attention_weight_seq = d2l.predict_seq2seq(
        net, eng, src_vocab, tgt_vocab, num_steps, device, True)
    print(f'{eng} => {translation}, ',
          f'bleu {d2l.bleu(translation, fra, k=2):.3f}')

In [ ]:
attention_weights = np.concatenate([step[0][0][0] for step in dec_attention_weight_seq], 0
    ).reshape((1, 1, -1, num_steps))

?????,????[**????????**]
???,??????????????????,???
???????,???????????????????????


In [ ]:
# ????????????
d2l.show_heatmaps(
    attention_weights[:, :, :, :len(engs[-1].split()) + 1],
    xlabel='Key positions', ylabel='Query positions')

## ??

* ??????,???????????????,????Bahdanau?????????????-???????????????????????????????????????????????
* ??????????-????,Bahdanau????????????????????,?????????????????????

## ??

1. ?????LSTM??GRU?
1. ?????????????????????????,??????????


[Discussions](https://discuss.d2l.ai/t/5753)
